# Train/Test Split for the Repair Experiments

To compare the repair strategies (R0–R4) fairly, queries are split into a **train** set (for developing strategies and learning the regression matrix) and a **test** set (held out for the final comparison). Tuning on everything would inflate the final numbers.

All cells run offline (no API calls) and produce real output.

**Properties:** balanced by difficulty, no leakage (train/test disjoint), and wired into the runner via `--splits`.

**Splits available:**

- **`subset_train` / `subset_test`** — the budget-sized subset we actually use: **60 train (20 easy / 20 medium / 20 human)** and **30 test (10 / 10 / 10)**, balanced equally across difficulty on both sides. Sized for DeepSeek V3. These UID lists are committed under `chinatravel/evaluation/default_splits/`; this notebook **reads and verifies** them (it does not regenerate them).
- **`pilot5`** — a shared 5-query set (2 easy / 1 medium / 2 human) everyone runs first, for a quick R0-vs-Rx comparison (Step 6). Its 5 queries are a **subset of `subset_train`**.
- **`combined_train` / `combined_test`** — the full 422/182 stratified split, if budget allows.

Model: **DeepSeek V3** (`deepseek/deepseek-chat`) via OpenRouter — cheap enough to run the experiments on a small budget.

## Step 1: What the split is

Queries come from the bilingual ChinaTravel dataset on Hugging Face, across three difficulty sets: `easy`, `medium`, `human`. Runs are in English, so each query's English text is used and city names are translated (handled in `chinatravel/data/load_datasets.py`).

Because of our API budget we don't run all ~600 queries. We use a **90-query balanced subset** — 30 of each difficulty, split 20 train / 10 test per difficulty. The UID lists live in `chinatravel/evaluation/default_splits/subset_train.txt` and `subset_test.txt`, and `chinatravel/evaluation/splits/subset_origin.json` records each UID's difficulty. The cell below loads those committed files.

In [1]:
import os, io, json, contextlib
from types import SimpleNamespace
from collections import Counter

# Run from the repo root so the chinatravel package imports resolve.
if os.path.basename(os.getcwd()) != 'Agentic-Travel-Planner-Ablation-of-Verifications-' \
        and os.path.isdir('Agentic-Travel-Planner-Ablation-of-Verifications-'):
    os.chdir('Agentic-Travel-Planner-Ablation-of-Verifications-')

SPLIT_DIR = 'chinatravel/evaluation/default_splits'
ORIGIN_PATH = 'chinatravel/evaluation/splits/subset_origin.json'

def read_ids(name):
    with open(f'{SPLIT_DIR}/{name}.txt', encoding='utf-8') as f:
        return [ln.strip() for ln in f if ln.strip()]

# The committed split is the single source of truth; we read it (never regenerate it).
train_all = read_ids('subset_train')
test_all  = read_ids('subset_test')
origin = json.load(open(ORIGIN_PATH, encoding='utf-8'))['origin']

print(f'subset_train.txt: {len(train_all)} uids')
print(f'subset_test.txt : {len(test_all)} uids')
print('origin map covers all split uids:',
      set(train_all + test_all) <= set(origin))

subset_train.txt: 60 uids
subset_test.txt : 30 uids
origin map covers all split uids: True


## Step 2: The split (balanced 20/10 per difficulty)

Each difficulty contributes 20 train and 10 test queries, so train and test carry the same difficulty mix and neither side is biased toward a difficulty. The lists are committed, so everyone works from the identical split.

In [2]:
# Per-difficulty counts in the committed split (difficulty from the origin map).
for name, ids in [('train', train_all), ('test', test_all)]:
    c = Counter(origin[u] for u in ids)
    print(f"{name}: " + ' / '.join(f'{k} {c[k]}' for k in ('easy', 'medium', 'human')))

print()
print('TOTAL train:', len(train_all), '| test:', len(test_all),
      '| total:', len(train_all) + len(test_all))
print('overlap (leakage):', set(train_all) & set(test_all) or 'none')

train: easy 20 / medium 20 / human 20
test: easy 10 / medium 10 / human 10

TOTAL train: 60 | test: 30 | total: 90
overlap (leakage): none


## Step 3: Check the difficulty mix is balanced

Both train and test should have equal easy/medium/human, so neither side is biased toward a difficulty.

In [3]:
def mix(ids):
    c = Counter(origin[u] for u in ids); t = len(ids)
    return {k: f'{v} ({100*v/t:.0f}%)' for k, v in sorted(c.items())}
print('train mix:', mix(train_all))
print('test  mix:', mix(test_all))

train mix: {'easy': '20 (33%)', 'human': '20 (33%)', 'medium': '20 (33%)'}
test  mix: {'easy': '10 (33%)', 'human': '10 (33%)', 'medium': '10 (33%)'}


## Step 4: How the split wires into the runner

A "split" in ChinaTravel is just a text file of query UIDs under `chinatravel/evaluation/default_splits/`. The train/test UID lists are `subset_train.txt` / `subset_test.txt`, with an origin map (`subset_origin.json`) recording each UID's difficulty so results can be sliced later.

The query text lives on Hugging Face (not local files), so `load_query` in `load_datasets.py` was extended: `--splits subset_train` loads the right base queries from Hugging Face and keeps only the UIDs in the file — no local query files needed.

The same wiring serves the full-dataset split (`combined_train` / `combined_test`, 422/182); the subset is just the budget-sized version. These files are the single source of truth — this notebook reads them rather than rewriting them, so re-running it can never clobber the committed split.

In [4]:
# This notebook does NOT write the split files — they are committed and
# treated as the source of truth. We only confirm the pilot5 relationship.
pilot5 = read_ids('pilot5')
print('pilot5:', len(pilot5), 'uids ->',
      ' / '.join(f'{k} {Counter(origin[u] for u in pilot5)[k]}'
                 for k in ('easy', 'medium', 'human')))
print('pilot5 is a subset of subset_train:', set(pilot5) <= set(train_all))

pilot5: 5 uids -> easy 2 / medium 1 / human 2
pilot5 is a subset of subset_train: True


## Step 5: Check the splits load through the runner's loader

Load `subset_train` and `subset_test` through the same `load_query` the experiment runner uses, and confirm the sizes, that they're disjoint (no leakage), and that a query comes through in English.

In [5]:
from chinatravel.data.load_datasets import load_query

def load(split):
    with contextlib.redirect_stdout(io.StringIO()):
        return load_query(SimpleNamespace(splits=split, lang='en', oracle_translation=True))

tr_ids, tr = load('subset_train')
te_ids, te = load('subset_test')
print('subset_train:', len(tr_ids))
print('subset_test :', len(te_ids))
print('disjoint (no leakage):', set(tr_ids).isdisjoint(te_ids))
print('train mix:', mix(tr_ids))
print('test  mix:', mix(te_ids))
print()
q = tr[tr_ids[0]]
print('example train query (English):', q['start_city'], '->', q['target_city'])
print('  ', q['nature_language'][:140])

subset_train: 60
subset_test : 30
disjoint (no leakage): True
train mix: {'easy': '20 (33%)', 'human': '20 (33%)', 'medium': '20 (33%)'}
test  mix: {'easy': '10 (33%)', 'human': '10 (33%)', 'medium': '10 (33%)'}

example train query (English): Shenzhen -> Beijing
   Current location: Shenzhen. I want to travel alone to Beijing for a day with a budget of 2,500 RMB. Please provide me with a travel plan.


## Step 6: Running R0 on the split (the experiment)

R0 runs on each query, saving one result file per query. The cells below show how one query is run, then loop over the sample printing each query's outcome and difficulty.

**Reproducibility.** R0 calls DeepSeek V3, which costs money and is non-deterministic. So results are **run once and committed** to `experiments/r0_deepseek-chat_pilot5/`. The loop is resume-safe: if a result exists it loads it instead of calling the API, so re-running the notebook reuses the committed files — no repeat cost, identical files for everyone.

Live calls need `RUN_LIVE = True` and an OpenRouter key in `.env` (defaults to `False`).

**Committed here:** R0 on the shared `pilot5` set. (Earlier GPT-4.1 samples are kept under `experiments/r0_combined_sample_gpt4.1/` and `experiments/r0_medium_sample_gpt4.1/` for reference.)

In [6]:
import os, io, json, contextlib
from types import SimpleNamespace

# Committed R0 results (tracked folder, unlike the git-ignored results/).
# These are the DeepSeek-V3 runs on the shared 5-query pilot set.
RESULTS_DIR = 'experiments/r0_deepseek-chat_pilot5'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Safety switch: keep False so the notebook never makes paid API calls.
RUN_LIVE = False
MAX_QUERIES = 5                 # cap for live runs; None = whole split
SPLIT = 'pilot5'               # shared 5-query pilot (subset_train for the fuller run)

REPAIR_STRATEGY = 'R0'
REFINE_STEPS = 5
LLM = 'deepseek/deepseek-chat'  # DeepSeek V3 via OpenRouter

# Difficulty of each query, for slicing results. pilot5 queries are a subset of
# the stratified subset, so the subset origin map covers them.
ORIGIN = json.load(open('chinatravel/evaluation/splits/subset_origin.json'))['origin']

def result_path(uid):
    return os.path.join(RESULTS_DIR, f'{uid}.json')

print('results dir :', RESULTS_DIR)
print('RUN_LIVE    :', RUN_LIVE, '| MAX_QUERIES:', MAX_QUERIES, '| SPLIT:', SPLIT)
print('strategy    :', REPAIR_STRATEGY, '| refine_steps:', REFINE_STEPS, '| llm:', LLM)

results dir : experiments/r0_deepseek-chat_pilot5
RUN_LIVE    : False | MAX_QUERIES: 5 | SPLIT: pilot5
strategy    : R0 | refine_steps: 5 | llm: deepseek/deepseek-chat


### 6.1 How one query is run

The recipe for a single query mirrors `run_exp.py`: build the agent once, call `agent.solve(query, ...)`, save the plan as `<uid>.json`. The equivalent command for the whole split:

```bash
python run_exp.py \
  --agent LLM-modulo \
  --repair_strategy R0 \
  --llm deepseek/deepseek-chat \
  --splits subset_train \
  --refine_steps 5 \
  --oracle_translation
```

The function below is the per-query version of that command.

In [7]:
def build_runtime():
    """Build the R0 agent runtime once (same as run_exp.py sets up)."""
    from chinatravel.agent.load_model import create_agent_runtime
    return create_agent_runtime(
        'LLM-modulo', LLM,
        project_root_path=os.getcwd(),
        lang='en', oracle_translation=True,
        refine_steps=REFINE_STEPS, repair_strategy=REPAIR_STRATEGY,
    )

def run_one_query(runtime, uid, query):
    """Run R0 on one query and save the plan. Returns the plan dict."""
    succ, plan = runtime.agent.solve(query, prob_idx=uid, oracle_verifier=True)
    with open(result_path(uid), 'w', encoding='utf-8') as f:
        json.dump(plan, f, ensure_ascii=False, indent=2)
    return plan

def summarise(uid, plan):
    """One-line outcome for a query's result."""
    errs = plan.get('error_info', [])
    pc = plan.get('personal_constraints', [])
    passed = (len(errs) == 0) and all(x is True for x in pc) if pc else (len(errs) == 0)
    return (f"{uid}: {'PASS' if passed else 'needs work'} | "
            f"reported_errors={len(errs)} | personal={pc}")

print('Helpers defined: build_runtime(), run_one_query(), summarise().')

Helpers defined: build_runtime(), run_one_query(), summarise().


### 6.2 Run (or load) each query

For each query the loop loads its committed result if present, or runs it live if `RUN_LIVE` is on. Each line shows the query's **difficulty**.

In [8]:
# Committed results drive the default (offline) view; the live path draws from SPLIT.
committed = sorted(f[:-5] for f in os.listdir(RESULTS_DIR) if f.endswith('.json'))

if RUN_LIVE:
    with contextlib.redirect_stdout(io.StringIO()):
        split_ids, split_data = load_query(
            SimpleNamespace(splits=SPLIT, lang='en', oracle_translation=True))
    # Run committed ones first (loaded), then fill up to MAX_QUERIES from the split.
    order = committed + [u for u in split_ids if u not in committed]
else:
    order, split_data = committed, {}

runtime = None
results = {}
n_loaded = n_ran = n_pending = 0

for i, uid in enumerate(order, start=1):
    diff = ORIGIN.get(uid, '?')
    if os.path.exists(result_path(uid)):
        with open(result_path(uid), encoding='utf-8') as f:
            plan = json.load(f)
        results[uid] = plan; n_loaded += 1
        print(f'[{i:>2}] loaded  [{diff:6}] {summarise(uid, plan)}')
    elif RUN_LIVE and n_ran < MAX_QUERIES:
        if runtime is None:
            with contextlib.redirect_stdout(io.StringIO()):
                runtime = build_runtime()
        with contextlib.redirect_stdout(io.StringIO()):
            plan = run_one_query(runtime, uid, split_data[uid])
        results[uid] = plan; n_ran += 1
        print(f'[{i:>2}] ran     [{diff:6}] {summarise(uid, plan)}')
    # (queries beyond the committed sample / cap are simply not shown)

print()
print(f'loaded from committed results : {n_loaded}')
print(f'ran live this session         : {n_ran}')

[ 1] loaded  [easy  ] e20241028160320866976: needs work | reported_errors=2 | personal=[True, True, True, True, True]
[ 2] loaded  [easy  ] e20241028160324401047: needs work | reported_errors=2 | personal=[True, True, True, True, True]
[ 3] loaded  [medium] e20241028160857444675: needs work | reported_errors=11 | personal=[True, True, False, True, True, True, False, True]
[ 4] loaded  [human ] h20241029143502525564: needs work | reported_errors=2 | personal=[True, False, False, True, True, True]
[ 5] loaded  [human ] h20241029143506461809: needs work | reported_errors=3 | personal=[False, False, True, True]

loaded from committed results : 5
ran live this session         : 0


### 6.3 Aggregate result, broken down by difficulty

A summary over the sample: overall pass rate and average errors, then the same split by difficulty. The difficulty breakdown is the interesting part — it shows whether R0 struggles more as queries get harder.

In [9]:
if results:
    def is_pass(plan):
        errs = plan.get('error_info', [])
        pc = plan.get('personal_constraints', [])
        return (len(errs) == 0) and (all(x is True for x in pc) if pc else True)

    total = len(results)
    n_pass = sum(1 for p in results.values() if is_pass(p))
    avg_err = sum(len(p.get('error_info', [])) for p in results.values()) / total
    print(f'queries with a result : {total}')
    print(f'passed                : {n_pass} ({100*n_pass/total:.0f}%)')
    print(f'avg reported errors   : {avg_err:.1f}')

    # Break the sample down by difficulty (easy/medium/human).
    print()
    print('by difficulty:')
    from collections import defaultdict
    buckets = defaultdict(list)
    for uid, plan in results.items():
        buckets[ORIGIN.get(uid, '?')].append(plan)
    for diff in ['easy', 'medium', 'human']:
        ps = buckets.get(diff, [])
        if not ps:
            continue
        passed = sum(1 for p in ps if is_pass(p))
        ae = sum(len(p.get('error_info', [])) for p in ps) / len(ps)
        print(f'  {diff:6}: {len(ps)} queries | passed {passed}/{len(ps)} | avg errors {ae:.1f}')
else:
    print('No committed results found. Set RUN_LIVE=True (with an API key) to produce them,')
    print('then commit experiments/r0_combined_sample/ so teammates get the same files.')

queries with a result : 5
passed                : 0 (0%)
avg reported errors   : 4.0

by difficulty:
  easy  : 2 queries | passed 0/2 | avg errors 2.0
  medium: 1 queries | passed 0/1 | avg errors 11.0
  human : 2 queries | passed 0/2 | avg errors 2.5


### 6.3b Reading the results

**"Pass" is all-or-nothing.** A plan passes only if it satisfies *every* rule — the user's personal requests (budget, cuisine, room type…) **and** the commonsense rules (times line up, transport costs/structure are right). One broken detail fails the whole plan.

**Difficulty trend.** Error counts tend to rise from easy → medium → human; harder, more interdependent queries are where repair matters most — the gap R1–R4 aim to close.

**Most failures are wrong transport cost.** The route itself is valid, but the model fills in the *fare* between places and often gets it wrong (e.g. writes 7 when the real fare is 8), which the verifier rejects. The cell below shows one such mismatch. A low pass rate is expected for the naive baseline — if R0 passed everything there'd be nothing to improve.

> Note: an earlier version of the verifier defaulted to the Chinese database even for English runs, which produced false "unavailable transport" errors. That was fixed (the verifier is now language-aware), so these counts reflect genuine violations.

In [10]:
import io, contextlib, json, glob
from collections import Counter
from chinatravel.environment.world_env import WorldEnv

# (1) What kind of error dominates across the committed results?
kinds, total = Counter(), 0
for p in glob.glob('experiments/r0_deepseek-chat_pilot5/*.json'):
    for e in json.load(open(p)).get('error_info', []):
        total += 1
        s = str(e)
        if 'Inner-City Transport' in s:
            kinds['transport unavailable'] += 1
        elif 'cost' in s.lower() and 'transport' in s.lower():
            kinds['transport wrong cost'] += 1
        else:
            kinds['other'] += 1
print('Error breakdown across the committed results:')
for k, v in kinds.most_common():
    print(f'  {k}: {v}  ({100*v/total:.0f}% of all {total} errors)')

# (2) One concrete example: a metro leg whose fare DOESN'T match the real data.
with contextlib.redirect_stdout(io.StringIO()):
    env = WorldEnv(lang='en')

def find_fare_mismatch():
    for p in sorted(glob.glob('experiments/r0_deepseek-chat_pilot5/*.json')):
        d = json.load(open(p)); city = d.get('target_city')
        for day in d.get('itinerary', []):
            for act in day.get('activities', []):
                legs = act.get('transports')
                if not isinstance(legs, list) or not act.get('position'):
                    continue
                metro = [l for l in legs if l.get('mode') == 'metro']
                if not metro:
                    continue
                try:
                    with contextlib.redirect_stdout(io.StringIO()):
                        real = env.transportation.goto(city, legs[0]['start'], act['position'], legs[0]['start_time'], 'metro')
                    rm = [l for l in real if l.get('mode') == 'metro']
                except Exception:
                    continue
                if rm and rm[0].get('cost') != metro[0].get('cost'):
                    return city, legs[0]['start'], act['position'], metro[0]['cost'], rm[0]['cost']
    return None

ex = find_fare_mismatch()
if ex:
    city, origin, dest, llm_cost, real_cost = ex
    print(f'\nExample of a wrong detail ({city}): {origin} -> {dest}')
    print(f'  model wrote metro fare : {llm_cost}')
    print(f'  real sandbox fare      : {real_cost}')
    print(f'\n  -> Same route, but the model guessed the fare ({llm_cost}) while the real')
    print(f'     fare is {real_cost}. The verifier compares against the real data, sees the')
    print("     mismatch, and rejects the leg. Enough rejected legs -> 'not passed'.")
else:
    print('\n(No fare mismatch in this sample; transport errors come from other guessed')
    print(' details such as timing or mode.)')

Error breakdown across the committed results:
  transport wrong cost: 13  (65% of all 20 errors)
  other: 7  (35% of all 20 errors)



Example of a wrong detail (Beijing): Beijing Daxing International Airport -> The Palace Museum
  model wrote metro fare : 7
  real sandbox fare      : 8

  -> Same route, but the model guessed the fare (7) while the real
     fare is 8. The verifier compares against the real data, sees the
     mismatch, and rejects the leg. Enough rejected legs -> 'not passed'.


### 6.4 Continuing with registered repair strategies

The committed `experiments/r0_deepseek-chat_pilot5/` files are historical DeepSeek V3, five-step results. This notebook can read them without model calls. New six-attempt runs use a different protocol; do not present the archived baseline as a matched new comparison.

`run_experiments.py` is the repair-study runner; `run_exp.py` remains the general ChinaTravel/baseline launcher.

```bash
python run_experiments.py --strategy R1 --splits pilot5 --max-steps 6 \
  --lang en --llm YOUR_PROVIDER_MODEL_ID --run-id pilot5-r1
python scripts/build_repair_summary.py --input results/experiments/pilot5-r1
python scripts/evaluate_repair_run.py --run-dir results/experiments/pilot5-r1
python scripts/build_regression_matrix.py \
  --input results/experiments/pilot5-r1/attempts.jsonl \
  --output-dir results/experiments/pilot5-r1/regression_matrix
```

Raw evidence lives in `results/experiments/<RUN_ID>/`. Use the same model, language, population, and budget across new comparisons. Supported strategies are R0/R1/R2/R3 (R4 has its own guarded pilot runner). Scale up with `--splits subset_train` and a new run ID. For a specific query use `--index UID`; this is explicit UID selection, not a first-N limit. Resume with the same arguments plus `--resume`, optionally `--retry-failed`.

See [current commands](experiments/Commands_to_test_queries.md) and [the evidence contract](chinatravel/evaluation/REPAIR_COMPARISON.md). Difficulty maps remain in `chinatravel/evaluation/splits/`.
